In [1]:
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account
from google.api_core.exceptions import Conflict

# 1️⃣ Authentication and Setup

PROJECT_ID = "trim-plexus-396409"
DATASET_ID = "hr1_dataset"
TABLE_ID = "employee_transformed2"
CSV_PATH = r"C:\SRC_FILES\emp_src_1.csv"

key_path = r"C:\trim-plexus-396409-6c69e030270a.json"  # change this path
credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=credentials, project=credentials.project_id)

# 2️⃣ Extract: Read CSV file into pandas
print("📥 Extracting data from CSV...")
df = pd.read_csv(CSV_PATH)

print("✅ Data extracted. Sample:")
print(df.head())

# 3️⃣ Transform: Clean and enrich data
print("🔄 Transforming data...")

# Example transformations:
# - Remove duplicates
# - Fill missing salaries with average
# - Create a new column for annual salary
# - Standardize department names

df = df.drop_duplicates()
df["salary"] = df["salary"].fillna(df["salary"].mean())
df["annual_salary"] = df["salary"] * 12
df["department"] = df["department"].str.strip().str.title()  # Clean up names

# Add new column with current load timestamp
from datetime import datetime
df["load_timestamp"] = datetime.utcnow()

print("✅ Transformation complete. Sample:")
print(df.head())

# 4️⃣ Load: Create dataset and table in BigQuery if not exists
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "US"

try:
    client.create_dataset(dataset_ref)
    print(f"✅ Dataset '{DATASET_ID}' created.")
except Conflict:
    print(f"ℹ️ Dataset '{DATASET_ID}' already exists.")

# Define schema for BigQuery table
schema = [
    bigquery.SchemaField("emp_id", "INTEGER"),
    bigquery.SchemaField("name", "STRING"),
    bigquery.SchemaField("salary", "FLOAT"),
    bigquery.SchemaField("department", "STRING"),
    bigquery.SchemaField("annual_salary", "FLOAT"),
    bigquery.SchemaField("load_timestamp", "TIMESTAMP"),
]

table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
table = bigquery.Table(table_ref, schema=schema)

try:
    client.create_table(table)
    print(f"✅ Table '{TABLE_ID}' created.")
except Conflict:
    print(f"ℹ️ Table '{TABLE_ID}' already exists.")

# 5️⃣ Load transformed data into BigQuery
print("🚀 Loading transformed data into BigQuery...")

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",  # Replace table data
)

job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
job.result()

print(f"✅ Successfully loaded {len(df)} rows into {table_ref}.")

📥 Extracting data from CSV...
✅ Data extracted. Sample:
   employee_id         name   department   salary  Annualsalary
0            1     John Doe  Engineering  75000.0     2000000.0
1            2   Jane Smith    Marketing  65000.0    30000000.0
2            3  Bob Johnson           HR  60000.0     4000000.0
3            4  Alice Brown  Engineering  80000.0    33333333.0
4            4  Alice Brown  Engineering  80000.0    33333333.0
🔄 Transforming data...
✅ Transformation complete. Sample:
   employee_id         name   department   salary  Annualsalary  \
0            1     John Doe  Engineering  75000.0     2000000.0   
1            2   Jane Smith    Marketing  65000.0    30000000.0   
2            3  Bob Johnson           Hr  60000.0     4000000.0   
3            4  Alice Brown  Engineering  80000.0    33333333.0   
5            3          Bob           Hr      7.0     4000000.0   

   annual_salary             load_timestamp  
0       900000.0 2025-10-30 06:14:26.238489  
1      

C:\Users\Rajesh\AppData\Local\Temp\ipykernel_8744\904116663.py:40: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df["load_timestamp"] = datetime.utcnow()


ℹ️ Dataset 'hr1_dataset' already exists.
✅ Table 'employee_transformed2' created.
🚀 Loading transformed data into BigQuery...


C:\Users\Rajesh\AppData\Local\Programs\Python\Python314\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:484: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


✅ Successfully loaded 7 rows into trim-plexus-396409.hr1_dataset.employee_transformed2.
